In [2]:
import pandas as pd
import duckdb
import os
import numpy as np


conn = duckdb.connect('../database/olist.db')

# JOINS

##### Write a query joining olist_orders and olist_customers to return all orders alongside the customer's state. Then explain — what happens to orders where the customer_id exists in orders but not in customers? Which JOIN type protects you from silent data loss here?

In [12]:
query = """ 

    select
        o.order_id, o.customer_id, o.order_status, o.order_purchase_timestamp,
        c.customer_state, c.customer_city,
        CASE 
            WHEN c.customer_id IS NULL then 'orphaned'
            else 'matched' 
        END as record_quality_flag
    from orders o
    LEFT JOIN customers c
    on o.customer_id = c.customer_id


"""

conn.execute(query).df()

,order_id,customer_id,order_status,order_purchase_timestamp,customer_state,customer_city,record_quality_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,SP,sao paulo,matched
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,BA,barreiras,matched
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,GO,vianopolis,matched
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,SP,santo andre,matched
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,PR,congonhinhas,matched
...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,SP,sao jose dos campos,matched
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,SP,praia grande,matched
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,BA,nova vicosa,matched
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,RJ,japuiba,matched


In [15]:
query = """ 

    -- How bad is the data quality problem?
    SELECT
        COUNT(*)                                        AS total_orders,
        COUNT(c.customer_id)                            AS matched_orders,
        COUNT(*) - COUNT(c.customer_id)                 AS orphaned_orders,
        ROUND(
            (COUNT(*) - COUNT(c.customer_id)) * 100.0 
            / COUNT(*), 2)                              AS orphaned_pct

    FROM orders o
    LEFT JOIN customers c
        ON o.customer_id = c.customer_id


"""

conn.execute(query).df()

,total_orders,matched_orders,orphaned_orders,orphaned_pct
0,99441,99441,0,0.0


##### Join olist_orders, olist_order_items and olist_products to return order_id, product_category_name and total price per order. A junior analyst used INNER JOIN throughout — what records are they silently losing and why does it matter?

In [20]:
query = """ 

    select
        o.order_id, count(p.product_category_name) as total_categories, round(sum(oi.price),2) as total_price
    from orders o
    LEFT JOIN order_items oi
    on o.order_id = oi.order_id
    LEFT JOIN products p
    on oi.product_id = p.product_id
    group by o.order_id

"""

conn.execute(query).df()

,order_id,total_categories,total_price
0,0028de0ca693a1bb26448916a81105cc,1,29.99
1,00611822267e76e0055c25c18506f06e,1,159.90
2,0068e836900867da359bd81db9227a33,1,96.99
3,006dd93155bc2abd844cc5eed3a0fe7f,1,49.90
4,00ab9fa91aafa3d881164b0da48999aa,1,579.00
...,...,...,...
99436,6d9bd9b4caa7d961a6d31427ad0e557e,0,NaN
99437,801bda1bc87e8484430a1ad75f51f128,0,NaN
99438,261e56a43de07e1b1bc9e8615c8e8aa9,0,NaN
99439,7018a2cb4802de2f9ed4b09b6a98e6fd,0,NaN


##### Return all sellers from olist_sellers who have never received any order in olist_order_items. Write this using three different approaches — LEFT JOIN with NULL check, NOT EXISTS and NOT IN. Explain which is safest and why.

In [ ]:
# Left join NULL CHECKS
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city, oi.order_id
    from sellers s
    LEFT JOIN order_items oi
    on s.seller_id = oi.seller_id
    WHERE oi.seller_id IS NULL
     

"""

conn.execute(query).df()
# NOT EXSITS
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city
    from sellers s
    where NOT EXISTS (
        SELECT 1
        FROM order_items oi
        where oi.seller_id = s.seller_id
    )
     

"""

conn.execute(query).df()
# NOT IN
query = """ 

    select
       s.seller_id, s.seller_state, s.seller_city
    from sellers s
    LEFT JOIN order_items oi
    on s.seller_id = oi.seller_id
    WHERE s.seller_id NOT IN (
        select seller_id
        from order_items
        where seller_id is not null
    )
     

"""

conn.execute(query).df()

,seller_id,seller_state,seller_city


##### Perform a FULL OUTER JOIN between olist_customers and olist_orders. Identify — customers with no orders, orders with no matching customer and explain under what real-world data scenarios each of these anomalies occurs.

In [42]:
query = """ 

    SELECT
        -- Coalesce to handle NULLs from either side
        COALESCE(c.customer_id, 'No Customer Record')   AS customer_id,
        COALESCE(o.order_id,    'No Order Record')       AS order_id,
        c.customer_city,
        c.customer_state,
        o.order_status,
        o.order_purchase_timestamp,

        -- Classify every row into its bucket
        CASE
            WHEN c.customer_id IS NULL THEN 'Orphaned Order'
            WHEN o.customer_id IS NULL THEN 'Customer Without Order'
            ELSE 'Matched Record'
        END                                             AS record_type

    FROM customers c
    FULL OUTER JOIN orders o
        ON c.customer_id = o.customer_id

    -- Optional: filter to only anomalies
    WHERE c.customer_id IS NULL
    OR o.customer_id IS NULL

"""

conn.execute(query).df()

,customer_id,order_id,customer_city,customer_state,order_status,order_purchase_timestamp,record_type


##### A colleague joins olist_orders to olist_order_payments using a LEFT JOIN and then filters on payment_type = 'credit_card' in the WHERE clause. The result silently converts the LEFT JOIN into an INNER JOIN. Explain exactly why this happens and rewrite it correctly.

In [56]:
query = """ 

    SELECT
       o.order_id, p.payment_type,
       CASE 
        when p.payment_type IS NULL then 'No Payment'
        when p.payment_type = 'credit_card' then 'Credit card order' 
            else 'Other Payment'
        END as record_type
    FROM orders o
    LEFT JOIN order_payments p
    on o.order_id = p.order_id
        and p.payment_type = 'credit_card'
"""

conn.execute(query).df()

,order_id,payment_type,record_type
0,e481f51cbdc54678b7cc49136f2d6af7,credit_card,Credit card order
1,47770eb9100c2d0c44946d9cf07ec65d,credit_card,Credit card order
2,949d5b44dbf5de918fe9c16f97b45f8a,credit_card,Credit card order
3,ad21c59c0840e6cb83a9ceb5573f8159,credit_card,Credit card order
4,a4591c265e18cb1dcee52889e2d8acc3,credit_card,Credit card order
...,...,...,...
99726,5fabc81b6322c8443648e1b21a6fef21,None,No Payment
99727,788541a19c0791de0504c5a9cb7e7bd5,None,No Payment
99728,f9e3402be5a5ea63344347582ca9f45f,None,No Payment
99729,38e9133ce29f6bbe35aed9c3863dce01,None,No Payment


##### You are joining olist_orders (5M rows) to olist_order_items (10M rows) to olist_order_payments (6M rows). After the join your result set has significantly more rows than olist_orders. Diagnose exactly what is happening, why it happens and write the corrected query. This is the fan trap — explain it clearly as you would to a junior analyst.

In [6]:
query = """ 

    with opayments as (
        select order_id, count(distinct payment_type) as payments_used
        from order_payments
        group by order_id
    ),
    oitems as (
        select order_id, sum(price) as total_price
        from order_items
        group by order_id
    )
    SELECT
        count(o.order_id) 
    FROM orders o
    LEFT JOIN oitems oi
    ON o.order_id = oi.order_id
    LEFT JOIN opayments op
    ON o.order_id = op.order_id
    
"""

conn.execute(query).df()

,count(o.order_id)
0,99441


# SELF JOINs & Hierarchical Problems

##### Using a SELF JOIN on olist_customers, find all pairs of customers who live in the same city and state. Return customer_id_1, customer_id_2, city and state. Make sure you don't return duplicate pairs i.e. (A,B) and (B,A).

In [ ]:
query = """ 

    select
        c1.customer_id as c1,
        c2.customer_id as c2,
        c1.customer_city as shared_city
    from customers c1
    JOIN customers c2
    on c1.customer_city = c2.customer_city
        and c1.customer_id < c2.customer_id
    
"""

conn.execute(query).df()

##### Using a SELF JOIN on olist_order_reviews, find all cases where the same customer left reviews with a score difference of 3 or more points across two different orders. This could signal highly inconsistent customer experience.

In [15]:
query = """ 

    with review_score as (
        select
            o1.order_id, o1.review_id, o1.review_score, o1.review_creation_date, o.customer_id
        
        from order_reviews o1
        INNER JOIN orders o
        ON o1.order_id = o.order_id

    )

    select
        o1.customer_id, o1.order_id as oid, o2.order_id as oid2,
        o1.review_id as rid1, o2.review_id as rid2, o1.review_score as score1, o2.review_score as score2,
        abs(o1.review_score - o2.review_score) as score_diff,
        o1.review_creation_date as date1, o2.review_creation_date as date2,

        -- Business context - which direction is the incosistency?
        CASE
            when o1.review_score > o2.review_score then 'Expereince Declined' 
            when o2.review_score > o1.review_score then 'Expereince Improved'
        END as inconsistency_direction

    from review_score o1
    JOIN review_score o2
    on o1.customer_id = o2.customer_id
        and o1.review_score < o2.review_score
        where abs(o1.review_score - o2.review_score) >=3
    order by score_diff desc
    
"""

conn.execute(query).df()

,customer_id,oid,oid2,rid1,rid2,score1,score2,score_diff,date1,date2,inconsistency_direction
0,aa5ca8afef8fe0f0ef822a9da12d2cf1,4fafa25ee16300a26f7ef92f3d15b58b,4fafa25ee16300a26f7ef92f3d15b58b,b3cd01e7b091cf09f9bd2fff507e7d7e,fc6645b34f5a407938a0b5722ad74258,1,5,4,2018-03-11 00:00:00,2018-02-23 00:00:00,Expereince Improved
1,77e7a543346c063e804af0d1c7829ee0,95287dc5df93f25b90309d5acddbe02e,95287dc5df93f25b90309d5acddbe02e,8b925ef92d48c9d0114ff75e42f166e0,b77ca9738c1cd42a731e437f29fd6f1e,1,5,4,2018-04-12 00:00:00,2018-04-04 00:00:00,Expereince Improved
2,bdced1e75dafe66bd70dc84af644bca7,78cf5dc2baadfbac2c47c6ef7c2a2282,78cf5dc2baadfbac2c47c6ef7c2a2282,5bdf704ce1edc91bc6c73abede903d1c,55d37f60f12bd5af8b2185dda9ef6dea,1,5,4,2018-01-24 00:00:00,2018-01-24 00:00:00,Expereince Improved
3,cce8a7da61bf170e0e1780b28b1e3ac6,edc0fe108d350a1a98a62888287bc6f2,edc0fe108d350a1a98a62888287bc6f2,b7fed5b2a0c119e3afe6745493068050,83cf3a96fd28237ec40e326adfc8c12a,1,5,4,2018-03-02 00:00:00,2018-02-25 00:00:00,Expereince Improved
4,3b437bb9a2bdf74dc640da169e25a15b,2e84df8fa5a2f0d5eab815d6c157b1f3,2e84df8fa5a2f0d5eab815d6c157b1f3,e13128391a71b69596a2d644a2ad0948,5c5dd872d53874c7b9d013cd72264a66,1,5,4,2017-10-25 00:00:00,2017-10-09 00:00:00,Expereince Improved
...,...,...,...,...,...,...,...,...,...,...,...
60,9084b0a94916b534ac6e626011092567,8c6a3fab1ed272b02f23bb7dc9061d9a,8c6a3fab1ed272b02f23bb7dc9061d9a,72906dc18cd7aaed26be40c859613c65,dc7a14d3e1984bfc11b484b841697096,1,4,3,2018-05-01 00:00:00,2018-05-01 00:00:00,Expereince Improved
61,cd7df89ab76adcfe3e1b4a36b042b081,1a3ccc1695931b2cb7a95065e0c81e4b,1a3ccc1695931b2cb7a95065e0c81e4b,208dfdaf3da06528069df7b1c100ad57,baa56bf0b94ee6f09cd85fba21271c86,1,4,3,2017-11-19 00:00:00,2017-10-28 00:00:00,Expereince Improved
62,afbfb235f3ea4986295c741b4d21ea32,9a19e859f5bd7a0bb22d64d35fa8b979,9a19e859f5bd7a0bb22d64d35fa8b979,452c58e6ab8609f53592745a76557ca8,aba7a67fc891cbfe0dcc4222ce3b0bf0,1,4,3,2017-12-05 00:00:00,2017-12-04 00:00:00,Expereince Improved
63,ef9e24d18ac44872aa39e9bb721f9880,f93a732712407c02dce5dd5088d0f47b,f93a732712407c02dce5dd5088d0f47b,0174caf0ee5964646040cd94e15ac95e,d6517e29a505ea8f3199c123e5442d93,1,4,3,2018-03-07 00:00:00,2018-02-25 00:00:00,Expereince Improved


##### Using a SELF JOIN on olist_order_items, identify products that were ordered together in the same order — i.e. appear in the same order_id. Return product_id_1, product_id_2 and co-occurrence count. Filter for pairs that appeared together more than 10 times.

In [22]:
query = """ 

    select
        o1.product_id as p1,
        o2.product_id as p2,
        count(o1.order_id) as orders
    from order_items o1
    JOIN order_items o2
    on o1.order_id = o2.order_id
        and o1.product_id < o2.product_id
    group by o1.product_id,
        o2.product_id
    HAVING COUNT(o1.order_id) > 10
    order by orders desc
    
"""

conn.execute(query).df()

,p1,p2,orders
0,05b515fdc76e888aada3c6d66c201dff,270516a3f41dc035aa87d220228f844c,100
1,36f60d45225e60c7da4558b070ce4b60,e53e557d5a159f5aa2c5e995dfdf244b,48
2,710b7c26b7a742f497bba45fab91a25f,a9d9db064d4afd4458eb3e139fe29167,36
3,62995b7e571f5760017991632bbfd311,ac1ad58efc1ebf66bfadc09f29bdedc0,36
4,2ef36e1cae01b86d0ff0a2f50ff2bd53,53759a2ecddad2bb87a079a1f1519f73,30
5,35afc973633aaeb6b877ff57b2793310,99a4788cb24856965c36a24e339b6058,30
6,308e4e21ae228a10f6370a243ae59995,90b58782fdd04cb829667fcc41fb65f5,30
7,18486698933fbb64af6c0a255f7dd64c,dbb67791e405873b259e4656bf971246,26
8,58efb9b638561ce132216a9a612513e2,872db866d615db59612ac933f43d6b22,25
9,ad4b5def91ac7c575dbdf65b5be311f4,e6b314a2236c162ede1a879f1075430f,24


##### Simulate a seller peer comparison using a SELF JOIN. For each seller, find all other sellers in the same state and compare their total revenue, average review score and on-time delivery rate. Return each seller alongside their state average for each metric and flag sellers who are below state average on all three as 'Underperformer'.

In [63]:
query = """ 
    WITH seller_perf AS (
    SELECT
        oi.seller_id,
        s.seller_state,
        SUM(oi.price + oi.freight_value)            AS revenue,
        AVG(r.review_score)                         AS avg_review_score,

        -- Calculate on-time delivery rate inside CTE
        AVG(CASE
                WHEN o.order_delivered_customer_date
                     <= o.order_estimated_delivery_date
                THEN 1.0 ELSE 0.0
            END)                                    AS on_time_delivery_rate

    FROM order_items oi
    INNER JOIN orders o
        ON oi.order_id = o.order_id
    INNER JOIN order_reviews r
        ON oi.order_id = r.order_id
    INNER JOIN sellers s
        ON oi.seller_id = s.seller_id

    GROUP BY
        oi.seller_id,
        s.seller_state
),
state_averages AS (
    SELECT
        seller_state,
        AVG(revenue)                AS state_avg_revenue,
        AVG(avg_review_score)       AS state_avg_review_score,
        AVG(on_time_delivery_rate)  AS state_avg_delivery_rate,
        COUNT(seller_id)            AS sellers_in_state

    FROM seller_perf
    GROUP BY seller_state
),
final AS (
    SELECT
        sp.seller_id,
        sp.seller_state,
        sa.sellers_in_state,

        -- Seller metrics
        ROUND(sp.revenue, 2)                AS seller_revenue,
        ROUND(sp.avg_review_score, 2)       AS seller_review_score,
        ROUND(sp.on_time_delivery_rate, 2)  AS seller_delivery_rate,

        -- State averages
        ROUND(sa.state_avg_revenue, 2)          AS state_avg_revenue,
        ROUND(sa.state_avg_review_score, 2)     AS state_avg_review_score,
        ROUND(sa.state_avg_delivery_rate, 2)    AS state_avg_delivery_rate,

        -- Underperformer flag
        CASE
            WHEN sp.revenue             < sa.state_avg_revenue
            AND  sp.avg_review_score    < sa.state_avg_review_score
            AND  sp.on_time_delivery_rate < sa.state_avg_delivery_rate
            THEN 'Underperformer'
            ELSE 'Within Benchmark'
        END                                 AS performance_flag

    FROM seller_perf sp
    INNER JOIN state_averages sa
        ON sp.seller_state = sa.seller_state
)

SELECT
    s1.seller_id,
    s1.seller_state,
    s1.revenue                          AS seller_revenue,
    s1.avg_review_score                 AS seller_review_score,
    s1.on_time_delivery_rate            AS seller_delivery_rate,
    AVG(s2.revenue)                     AS state_avg_revenue,
    AVG(s2.avg_review_score)            AS state_avg_review_score,
    AVG(s2.on_time_delivery_rate)       AS state_avg_delivery_rate,

    CASE
        WHEN s1.revenue < AVG(s2.revenue)
        AND  s1.avg_review_score < AVG(s2.avg_review_score)
        AND  s1.on_time_delivery_rate < AVG(s2.on_time_delivery_rate)
        THEN 'Underperformer'
        ELSE 'Within Benchmark'
    END                                 AS performance_flag

FROM seller_perf s1
JOIN seller_perf s2
    ON  s1.seller_state = s2.seller_state
    AND s1.seller_id <> s2.seller_id    -- exclude self comparison

GROUP BY
    s1.seller_id,
    s1.seller_state,
    s1.revenue,
    s1.avg_review_score,
    s1.on_time_delivery_rate

ORDER BY performance_flag DESC
    
"""

conn.execute(query).df()

,seller_id,seller_state,seller_revenue,seller_review_score,seller_delivery_rate,state_avg_revenue,state_avg_review_score,state_avg_delivery_rate,performance_flag
0,0db783cfcd3b73998abc6e10e59a102f,SP,11024.550022,4.213235,0.963235,5518.283964,3.945755,0.841991,Within Benchmark
1,1da3aeb70d7989d1e6d9b0e887f97c23,SP,13053.420078,4.073171,0.942073,5517.184306,3.945831,0.842002,Within Benchmark
2,8d956fec2e4337affcb520f56fd8cbfd,SP,14387.590078,4.192140,0.895197,5516.461178,3.945767,0.842028,Within Benchmark
3,6a51fc556dab5f766ced6fbc860bc613,SP,8370.249878,3.939394,0.878788,5519.722609,3.945904,0.842037,Within Benchmark
4,ad273c9eb54ecb4532de4bf75bae9e4e,PR,829.750004,3.818182,1.000000,4195.469851,4.010307,0.890454,Within Benchmark
...,...,...,...,...,...,...,...,...,...
3080,ab75b89cc49c9ab3160d0c91565a442a,SP,53.830000,1.500000,0.500000,5524.230154,3.947226,0.842242,Underperformer
3081,8e4f041ff58e7845456d3482524014b3,RS,182.390007,3.000000,0.500000,3399.254210,4.030066,0.895001,Underperformer
3082,880eda903e719a5f179f7e9fceb3a69d,SP,291.720001,1.000000,0.000000,5524.101216,3.947497,0.842513,Underperformer
3083,8d899e15a5925f097cca50faa49b15e3,SP,795.979996,2.500000,0.400000,5523.827905,3.946684,0.842296,Underperformer


In [ ]:
()